In [2]:
import os
import torch
import torchaudio
import numpy as np
import pandas as pd
from torch.nn.utils.rnn import pad_sequence

# ============================
# CONFIG
# ============================
TARGET_SR = 16000
N_MFCC = 40
MAX_FRAMES = 600
OUTPUT_FILE = "mfcc_lstm_sequences.npz"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# ============================
# TORCHAUDIO MFCC TRANSFORM
# ============================
mfcc_transform = torchaudio.transforms.MFCC(
    sample_rate=TARGET_SR,
    n_mfcc=N_MFCC,
    melkwargs={
        "n_fft": 400,
        "hop_length": 160,
        "n_mels": 64
    }
).to(device)

# ============================
# INPUT DATAFRAME
# Must contain: audio_path, lang
# ============================
df = pd.read_csv("voxpopuli_balanced_metadata.csv")  # Adjust the path as needed

all_sequences = []
all_labels = []

print("\n==============================")
print("Starting MFCC extraction...")
print("==============================\n")

# ============================
# MAIN EXTRACTION LOOP
# ============================
for idx, row in df.iterrows():
    path = row["audio_path"]
    lang = row["lang"]

    print(f"\n▶ Extracting MFCC for file {idx+1}/{len(df)}")
    print(f"   Path   : {path}")
    print(f"   Lang   : {lang}")

    try:
        # 1. Load audio
        waveform, sr = torchaudio.load(path)

        # 2. Resample if needed
        if sr != TARGET_SR:
            resampler = torchaudio.transforms.Resample(sr, TARGET_SR)
            waveform = resampler(waveform)

        waveform = waveform.to(device)

        # 3. Compute MFCC
        mfcc = mfcc_transform(waveform)   # shape: (1, 40, T)
        mfcc = mfcc.squeeze(0)            # (40, T)
        mfcc = mfcc.transpose(0, 1)       # (T, 40)

        print(f"   MFCC shape BEFORE truncate: {mfcc.shape}")

        # 4. Truncate long sequences
        if mfcc.shape[0] > MAX_FRAMES:
            mfcc = mfcc[:MAX_FRAMES]

        print(f"   MFCC shape AFTER truncate : {mfcc.shape}")

        all_sequences.append(mfcc.cpu())
        all_labels.append(lang)

    except Exception as e:
        print(f"   ❌ ERROR processing file {path}: {e}")
        print("   → Inserting zero-padding instead")

        all_sequences.append(torch.zeros((MAX_FRAMES, N_MFCC)))
        all_labels.append(lang)

# ============================
# PADDING ALL SEQUENCES
# ============================
print("\nPadding sequences to fixed length...")
padded_sequences = pad_sequence(all_sequences, batch_first=True)

print("✔ FINAL padded data shape:", padded_sequences.shape)
print("   (num_samples, max_frames, mfcc_dim)")
print("   =", padded_sequences.shape)

# ============================
# SAVE TO NPZ FILE
# ============================
np.savez_compressed(
    OUTPUT_FILE,
    X=padded_sequences.numpy(),
    y=np.array(all_labels)
)

print("\n=======================================")
print("✔ MFCC extraction COMPLETED successfully")
print(f"✔ Saved to: {OUTPUT_FILE}")
print("=======================================\n")

Using device: cuda

Starting MFCC extraction...


▶ Extracting MFCC for file 1/15000
   Path   : G:/.shortcut-targets-by-id/1I_1rgpq7N-oeJEqICjkPjQKmVMhQk48O/voxpopuli_data/it/batch_468/audio_4.wav
   Lang   : it
   MFCC shape BEFORE truncate: torch.Size([242, 40])
   MFCC shape AFTER truncate : torch.Size([242, 40])

▶ Extracting MFCC for file 2/15000
   Path   : G:/.shortcut-targets-by-id/1I_1rgpq7N-oeJEqICjkPjQKmVMhQk48O/voxpopuli_data/fr/batch_179/audio_7.wav
   Lang   : fr
   MFCC shape BEFORE truncate: torch.Size([676, 40])
   MFCC shape AFTER truncate : torch.Size([600, 40])

▶ Extracting MFCC for file 3/15000
   Path   : G:/.shortcut-targets-by-id/1I_1rgpq7N-oeJEqICjkPjQKmVMhQk48O/voxpopuli_data/it/batch_408/audio_4.wav
   Lang   : it
   MFCC shape BEFORE truncate: torch.Size([569, 40])
   MFCC shape AFTER truncate : torch.Size([569, 40])

▶ Extracting MFCC for file 4/15000
   Path   : G:/.shortcut-targets-by-id/1I_1rgpq7N-oeJEqICjkPjQKmVMhQk48O/voxpopuli_data/de/batch_107/audi